# ViGaL tests (static Q&A)

Load [ViGaL-7B](https://huggingface.co/yunfeixie/ViGaL-7B) (Qwen2.5-VL RL finetune) and ask it about a **pixel board**. Inspired by `notebooks/play.ipynb` (Ask box, live image, scrollback, generating lock) but **not** wired through NAMS / Gemma / `InteractiveSession`.

The board is an RGB image only, always resized to **512×512**. No ASCII grid, no settings JSON, no coordinate dump. Official Snake / Rotation presets are the paper Appendix A.2 *instructions* (rules + required tags); live positions are left out on purpose.

Reply format the weights were trained to emit:

- Snake: `<think>…</think><best_answer>UP|DOWN|LEFT|RIGHT</best_answer><worst_answer>…|None</worst_answer>`
- Rotation: `<think>…</think><answer>counter clockwise 90|180</answer>`

One **Ask** is one generation on the current image(s) + the question box.
**New board** draws a random 10×10 Snake scene (green = you, blue = enemy,
red = apples) as a 512×512 image — you do not need screenshots. Path /
upload / HF sample are optional extras.

Prereqs:

- `bash scripts/setup_env.sh`
- `bash game_alternative_research/vigal_setup.sh` (installs `qwen-vl-utils`, caches weights, fetches Rotation samples)

Do not keep `play.ipynb`'s Gemma loaded on the same GPU. The first cell downloads / loads ~8B BF16 weights and can take a minute.


In [ ]:
import os
import sys
from pathlib import Path

cwd = Path.cwd()
if cwd.name in ("notebooks", "game_alternative_research"):
    os.chdir(cwd.parent)
REPO = Path.cwd()
sys.path.insert(0, str(REPO / "game_alternative_research"))

from vigal_io import load_repo_env, load_vigal_model, MODEL_ID

load_repo_env()
print("WARNING: unload play.ipynb Gemma first if it is still on this GPU.")
print("Loading", MODEL_ID, "(first run is slow)...")
model, processor = load_vigal_model()
print("ready. device:", next(model.parameters()).device)


In [ ]:
import html as _html
from pathlib import Path

import ipywidgets as widgets
from IPython.display import clear_output, display

from vigal_io import (
    SNAKE_INSTRUCTION,
    ROTATION_INSTRUCTION,
    BOARD_SIZE,
    default_rotation_paths,
    generate_vigal,
    image_png_bytes,
    load_board_image,
    load_hf_sample_images,
    parse_vigal_reply,
    random_snake_board,
)

FRAME_WIDTH = 256
_images = []
_generating = False


def _official_text(kind):
    if kind == "Official Rotation":
        return ROTATION_INSTRUCTION
    return SNAKE_INSTRUCTION


preset = widgets.Dropdown(
    options=["Official Snake", "Official Rotation", "Custom"],
    value="Official Snake",
    description="Preset:",
    layout=widgets.Layout(width="360px"),
)
new_board_btn = widgets.Button(description="New board", button_style="success")
path_box = widgets.Text(
    value="",
    placeholder="optional: PNG path, comma-separated for Rotation",
    description="Path:",
    layout=widgets.Layout(width="640px"),
)
hf_index = widgets.IntText(value=0, description="HF row:", layout=widgets.Layout(width="200px"))
load_path_btn = widgets.Button(description="Load path")
load_rot_btn = widgets.Button(description="Load Rotation samples")
load_hf_btn = widgets.Button(description="Load HF sample")
upload = widgets.FileUpload(accept="image/*", multiple=True, description="Upload")
fill_btn = widgets.Button(description="Fill official prompt")
question_box = widgets.Textarea(
    value=SNAKE_INSTRUCTION,
    placeholder="Question / instruction...",
    description="You:",
    layout=widgets.Layout(width="720px", height="220px"),
)
ask_btn = widgets.Button(description="Ask", button_style="primary")
restart_btn = widgets.Button(description="Restart conversation", button_style="warning")
banner = widgets.HTML()
live = widgets.HBox()
out = widgets.Output(layout=widgets.Layout(width="100%"))


def _upload_items():
    val = upload.value
    if val is None:
        return []
    if isinstance(val, dict):
        return list(val.values())
    return list(val)


def _file_bytes(item):
    if isinstance(item, dict):
        return item.get("content") or item.get("content", b"")
    return getattr(item, "content", None)


def _sync_buttons():
    has = bool(_images)
    ask_btn.disabled = _generating or not has
    new_board_btn.disabled = _generating
    load_path_btn.disabled = _generating
    load_rot_btn.disabled = _generating
    load_hf_btn.disabled = _generating
    fill_btn.disabled = _generating
    upload.disabled = _generating
    restart_btn.disabled = _generating
    question_box.disabled = _generating


def _show_live():
    kids = []
    for i, im in enumerate(_images):
        kids.append(widgets.Image(value=image_png_bytes(im), width=FRAME_WIDTH))
        kids.append(widgets.HTML(f"<div style='font-family:monospace'>{i + 1}/{len(_images)}  {BOARD_SIZE}×{BOARD_SIZE}</div>"))
    live.children = tuple(kids) if kids else (
        widgets.HTML("<i>No board — press New board.</i>"),
    )
    _sync_buttons()


def _set_images(imgs, note=""):
    global _images
    _images = [load_board_image(im) for im in imgs]
    _show_live()
    if note:
        with out:
            print(note, f"n={len(_images)} at {BOARD_SIZE}×{BOARD_SIZE}")


def _refresh_banner(parsed=None):
    if not parsed:
        banner.value = (
            "<div style='padding:6px 10px;border:1px solid #888;background:#f7f7f7;"
            "font-family:monospace'><b>best</b>=(none) &nbsp; <b>worst</b>=(none) "
            "&nbsp; <b>answer</b>=(none)</div>"
        )
        return
    def esc(v):
        return _html.escape(v) if v else "(none)"
    banner.value = (
        "<div style='padding:6px 10px;border:1px solid #888;background:#f7f7f7;"
        "font-family:monospace'>"
        f"<b>best</b>={esc(parsed.get('best_answer'))}"
        f" &nbsp; <b>worst</b>={esc(parsed.get('worst_answer'))}"
        f" &nbsp; <b>answer</b>={esc(parsed.get('answer'))}"
        "</div>"
    )


def on_preset(_=None):
    if preset.value != "Custom":
        question_box.value = _official_text(preset.value)


def on_fill(_):
    if preset.value == "Custom":
        return
    question_box.value = _official_text(preset.value)


def on_new_board(_=None):
    preset.value = "Official Snake"
    question_box.value = SNAKE_INSTRUCTION
    _set_images([random_snake_board()], "New random Snake board.")


def on_load_path(_):
    raw = path_box.value.strip()
    if not raw:
        return
    paths = [Path(p.strip()) for p in raw.split(",") if p.strip()]
    missing = [p for p in paths if not p.is_file()]
    if missing:
        with out:
            print("missing:", ", ".join(str(p) for p in missing))
        return
    _set_images(paths, "Loaded path(s).")


def on_load_rot(_):
    paths = default_rotation_paths()
    missing = [p for p in paths if not p.is_file()]
    if missing:
        with out:
            print("Rotation samples missing; run bash game_alternative_research/vigal_setup.sh")
            print("missing:", ", ".join(str(p) for p in missing))
        return
    preset.value = "Official Rotation"
    question_box.value = ROTATION_INSTRUCTION
    _set_images(paths, "Loaded official Rotation samples.")


def on_load_hf(_):
    try:
        imgs = load_hf_sample_images(int(hf_index.value))
    except Exception as e:
        with out:
            print("HF sample failed:", type(e).__name__, e)
        return
    _set_images(imgs, f"Loaded HF vigal_data images from row {hf_index.value} (pixels only).")


def on_upload(_):
    items = _upload_items()
    if not items:
        return
    imgs = []
    for item in items:
        data = _file_bytes(item)
        if data:
            imgs.append(load_board_image(data))
    if imgs:
        _set_images(imgs, "Loaded upload(s).")


def _ask():
    q = question_box.value.strip()
    if not q or not _images:
        return
    print("=== You ===")
    print(q[:400] + ("…" if len(q) > 400 else ""))
    print(f"[{len(_images)} image(s) at {BOARD_SIZE}×{BOARD_SIZE}]")
    raw = generate_vigal(model, processor, _images, q)
    parsed = parse_vigal_reply(raw)
    print("\n=== ViGaL ===")
    print(raw)
    print(
        f"\n[best={parsed['best_answer']!r}  worst={parsed['worst_answer']!r}  "
        f"answer={parsed['answer']!r}]"
    )
    _refresh_banner(parsed)
    if preset.value != "Custom":
        question_box.value = _official_text(preset.value)


def on_ask(_):
    global _generating
    if _generating or not _images or not question_box.value.strip():
        return
    _generating = True
    _sync_buttons()
    try:
        with out:
            _ask()
    finally:
        _generating = False
        _sync_buttons()


def on_restart(_):
    with out:
        clear_output()
        print("Conversation scrollback cleared. Images kept.")
    _refresh_banner()


preset.observe(on_preset, names="value")
fill_btn.on_click(on_fill)
new_board_btn.on_click(on_new_board)
load_path_btn.on_click(on_load_path)
load_rot_btn.on_click(on_load_rot)
load_hf_btn.on_click(on_load_hf)
upload.observe(on_upload, names="value")
ask_btn.on_click(on_ask)
restart_btn.on_click(on_restart)

_refresh_banner()
on_new_board()
display(
    widgets.VBox([
        banner,
        live,
        widgets.HBox([new_board_btn, preset, fill_btn]),
        question_box,
        widgets.HBox([ask_btn, restart_btn]),
        widgets.HTML("<i>Optional: load a file instead of New board</i>"),
        path_box,
        widgets.HBox([load_path_btn, load_rot_btn, hf_index, load_hf_btn, upload]),
        out,
    ])
)
